# Voz de Asuka para Athena (Piper, entrenada en Kaggle gratis)

1. Arriba a la derecha: **Settings → Accelerator → GPU T4 x1** e **Internet → On** (hay que tener el móvil verificado en Kaggle).
2. **Add Input → Upload → Dataset**: sube `voz-asuka-datos.zip` (lo dejó Athena en Descargas). Nombre: `voz-asuka-datos`.
3. **Run All**. Tarda unas 5-6 horas; puedes cerrar la pestaña si usas **Save Version → Save & Run All (Commit)**.
4. Al terminar, en **Output** descarga `es_ES-asuka-medium.onnx` y `es_ES-asuka-medium.onnx.json` y mándaselos a Athena por Telegram (o déjalos en Descargas del portátil y dile «pon la voz de Asuka»).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!apt-get -qq update > /dev/null && apt-get -qq install -y espeak-ng build-essential cmake ninja-build > /dev/null
!git clone -q https://github.com/OHF-voice/piper1-gpl.git /kaggle/working/piper1-gpl
%cd /kaggle/working/piper1-gpl
!pip -q install -e '.[train]'
!./build_monotonic_align.sh > /dev/null
!python3 setup.py build_ext --inplace > /dev/null
print("listo")

In [ ]:
import glob, os
from huggingface_hub import hf_hub_download
csv = glob.glob('/kaggle/input/**/metadata.csv', recursive=True)[0]
audio = os.path.join(os.path.dirname(csv), 'wavs')
print(csv, len(open(csv).read().splitlines()), 'frases;', len(os.listdir(audio)), 'audios')
ck = hf_hub_download('rhasspy/piper-checkpoints', 'es/es_ES/davefx/medium/epoch=5629-step=1605020.ckpt',
                     repo_type='dataset', local_dir='/kaggle/working/base')
os.makedirs('/kaggle/working/asuka', exist_ok=True)
os.environ.update(CSV=csv, AUDIO=audio, CK=ck)
CSV, AUDIO, CK = csv, audio, ck

In [ ]:
# ~0,5 h de audio: 1.200 épocas desde una voz española ya entrenada (unas 5 h en una T4)
!python3 -m piper.train fit \
  --data.voice_name asuka \
  --data.csv_path "$CSV" \
  --data.audio_dir "$AUDIO" \
  --model.sample_rate 22050 \
  --data.espeak_voice es \
  --data.cache_dir /kaggle/working/cache \
  --data.config_path /kaggle/working/asuka/config.json \
  --data.batch_size 24 \
  --model.warmstart_ckpt "$CK" \
  --trainer.max_epochs 1200 \
  --trainer.accelerator gpu --trainer.devices 1 --trainer.precision 32 \
  --trainer.default_root_dir /kaggle/working/entreno

In [ ]:
import glob, os, shutil
ult = sorted(glob.glob('/kaggle/working/entreno/**/*.ckpt', recursive=True), key=os.path.getmtime)[-1]
print('checkpoint:', ult)
!python3 -m piper.train.export_onnx --checkpoint "{ult}" --output-file /kaggle/working/es_ES-asuka-medium.onnx
shutil.copy('/kaggle/working/asuka/config.json', '/kaggle/working/es_ES-asuka-medium.onnx.json')
!ls -la /kaggle/working/*.onnx*

In [ ]:
# Prueba: escucha cómo suena
!pip -q install piper-tts
from piper import PiperVoice
import wave, IPython
v = PiperVoice.load('/kaggle/working/es_ES-asuka-medium.onnx')
with wave.open('/kaggle/working/prueba.wav', 'wb') as w:
    v.synthesize_wav('Hola, soy Athena. Desde hoy hablo con la voz de Asuka. ¿Qué te parece?', w)
IPython.display.Audio('/kaggle/working/prueba.wav')